In [ ]:
from rdflib import Graph

g_strat = Graph()
g_strat.parse("/home/d-ladesteijn/p2f/p2f-api/p2f_api/adminutils/skos/stratigraphy.org.ttl", format="turtle")

print(f"Number of rows: {len(g_strat)}")
for text in g_strat:
    print(text)

In [ ]:
import pandas as pd

df_strat = pd.DataFrame(
    [(str(subject), str(predicate), str(obj)) for subject, predicate, obj in g_strat],
    columns=["subject","predicate", "object"]
)

print(df_strat)
df_strat.to_parquet("strat_full_data.parquet", index=False)

In [ ]:
import pandas as pd

rows = []

for subject, predicate, obj in g_strat:
    rows.append({
        "subject": str(subject),
        "object": str(obj),
        "object_type": type(obj).__name__,
        "object_lang": getattr(obj, "language", None),
    })

df_strat = pd.DataFrame(rows)
print(df_strat)

In [ ]:
from rdflib import Literal

languages = sorted({
    obj.language
    for _, _, obj in g_strat
    if isinstance(obj, Literal) and obj.language
})

print(languages)

In [ ]:
filtered_df_strat = df_strat.loc[
    df_strat["subject"].str.startswith("http://www.eionet.europa.eu", na=False)
    & df_strat["object_type"].eq("Literal")
    & df_strat["object_lang"].isin(["en", "en-US"])
].copy()

print(filtered_df_strat)

filtered_df_strat.to_parquet("strat_filtered_data.parquet", index=False)